<a href="https://colab.research.google.com/github/Selvapriya05/Selvapriya-Codeboosters-2026/blob/main/Day_3/Day_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests --quiet
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')
print(f'pandas  : {pd.__version__}')
print(f'requests: {requests.__version__}')


#Step-2 - EXTRACT: Load raw messy sales data

raw_df = pd.read_csv('messy_sales_data.csv')

print(f'Raw data loaded: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns')
print(f'columns: {raw_df.columns.tolist()}')
print('\nFirst 5 rows:')
raw_df.head()


All libraries imported successfully!
pandas  : 2.2.2
requests: 2.32.4
Raw data loaded: 30 rows, 9 columns
columns: ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']

First 5 rows:


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [ ]:
#Step-3 - Diagnose ALL data quality problems before fixing

print('=' * 55)
print('  DATA QUALITY DIAGNOSIS REPORT')
print('=' * 55)

#1. Missing values
print('\n[1] MISSING VALUES per column:')
print(raw_df.isnull().sum())

#2. Duplicates
print(f'\n[2] DUPLICATE ROWS : {raw_df.duplicated().sum()}')

#3. Data Types
print('\n[3] DATA TYPES:')
print(raw_df.dtypes)

#4. Unique values in text columns (spot inconsistencies)
print('\n[4] UNIQUE CATEGORIES:', raw_df['category'].unique())
print('[4] Sample customer names:', raw_df['customer_name'].dropna().unique()[:8])
print('[4] Sample order_date values:', raw_df['order_date'].unique()[:6])


  DATA QUALITY DIAGNOSIS REPORT

[1] MISSING VALUES per column:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATE ROWS : 0

[3] DATA TYPES:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATEGORIES: ['Electronics' 'Accessories' nan]
[4] Sample customer names: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']
[4] Sample order_date values: ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024'
 '2024-01-12']


In [ ]:
#Step-4 - Create a working copy (ETL best practice)

df = raw_df.copy()

print(f'Working copy created: {df.shape}')
print('raw_df is untouched - we can always reset by running df = raw_dff.copy()')


Working copy created: (30, 9)
raw_df is untouched - we can always reset by running df = raw_dff.copy()


In [ ]:
#Step-5 - Fix#1: Handle Missing values

print('Before fixing nulls:', df.isnull().sum().sum(), 'total missing values')

df['customer_name'].fillna('Unknown Customer',inplace=True)
median_qty = df['quantity'].median()
df['quantity'].fillna(median_qty,inplace=True)
print(f'  Filled missing quantity with median :  {median_qty}')

df['category'].fillna('Uncategorized', inplace=True)

print('After fixing nulls:', df.isnull().sum().sum(), 'total missing values')



Before fixing nulls: 7 total missing values
  Filled missing quantity with median :  2.0
After fixing nulls: 1 total missing values


In [ ]:
#Step-6 - Fix #2: Remove Duplicates

print(f'Before deduplication: {len(df)} rows')
print(f'Duplicate rows: {df.duplicated().sum()}')
print('\nDuplicate rows:')
print(df[df.duplicated(keep=False)][['order_id','customer_name','product','order_date']])

df.drop_duplicates(inplace=True)

print(f'After deduplication: {len(df)} rows')
print(f'Rows removed: {len(raw_df) - len(df)}')



Before deduplication: 30 rows
Duplicate rows: 0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []
After deduplication: 30 rows
Rows removed: 0


In [ ]:
print('Sample dates before parsing')
print(df['order_date'].head(8).tolist())

df['order_date'] = pd.to_datetime(
    df['order_date'],
    dayfirst=False,
    errors='coerce'
)

nat_count=df['order_date'].isnull().sum()
print(f'\nUnparseable dates (NaT): {nat_count}')

df['year'] = df['order_date'].dt.year
df['month'] = df['order_date'].dt.month
df['month_name'] = df['order_date'].dt.strftime('%B')  # B(month name) - A(showing week names)

print('\nSample dates after parsing')
print(df[['order_date','year','month','month_name']].head(5))



Sample dates before parsing
[Timestamp('2024-01-05 00:00:00'), Timestamp('2024-01-07 00:00:00'), Timestamp('2024-01-08 00:00:00'), Timestamp('2024-01-10 00:00:00'), Timestamp('2024-01-05 00:00:00'), Timestamp('2024-01-12 00:00:00'), Timestamp('2024-01-13 00:00:00'), Timestamp('2024-01-15 00:00:00')]

Unparseable dates (NaT): 0

Sample dates after parsing
  order_date  year  month month_name
0 2024-01-05  2024      1    January
1 2024-01-07  2024      1    January
2 2024-01-08  2024      1    January
3 2024-01-10  2024      1    January
4 2024-01-05  2024      1    January


In [ ]:
#Step-8 - Text Standardization + Wrong category

import pandas as pd

# Example dataset
data = {
    'customer_name': [' priya ', 'RAHUL', 'kavi'],
    'product': ['Mouse', 'Keyboard', 'Charger'],
    'category': ['Electronics', 'Accessories', 'Electronics']
}

# Create dataframe
df = pd.DataFrame(data)


# Before Standardization
print('Before Standardization:', df['customer_name'].unique()[:6])

# Standardize customer names
df['customer_name'] = (
    df['customer_name']
    .str.strip()
    .str.title()
)

# After Standardization
print('After Standardization:', df['customer_name'].unique()[:6])

# Create wrong category mask
wrong_mask = df['category'] != 'Accessories'

# Show wrong categories
print(df[wrong_mask][['product', 'category']])

# Fix wrong categories
df.loc[wrong_mask, 'category'] = 'Accessories'

# Final output
print('After fix: Unique Categories:', df['category'].unique()[:6])


Before Standardization: [' priya ' 'RAHUL' 'kavi']
After Standardization: ['Priya' 'Rahul' 'Kavi']
   product     category
0    Mouse  Electronics
2  Charger  Electronics
After fix: Unique Categories: ['Accessories']


In [ ]:
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').astype(int)
df['unitprice'] = pd.to_numeric(df['unit_price'], errors='coerce')
df['revenue'] =  df['quantity'] * df['unitprice']

print("Revenue column created")
print(f'\nTotal Revenue across all orders: {df["revenue"].sum}')


Revenue column created

Total Revenue across all orders: <bound method Series.sum of 0      90000
1      15000
2       3600
3      44000
4      90000
6       7000
7       2500
8      45000
9       6000
10     44000
11      4800
12     90000
14      5000
15      8000
16      6000
17     22000
18     45000
19      9000
20      7000
21      7500
22     45000
23      4800
24      4800
25      3500
26     44000
27    135000
28      6000
29      5000
Name: revenue, dtype: int64>


In [ ]:
#Step-10 - Validate data final

print('='* 55)
print('   POST-CLEANING VALIDATION REPORT')
print('='* 55)

print(f'Orriginal rows   : {len(raw_df)}')
print(f'cleaned rows     : {len(df)}')
print(f'Rows removed     : {len(raw_df) - len(df)}') ()


In [ ]:
# Convert columns to numeric
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')

df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')

# Create revenue column
df['revenue'] = df['quantity'] * df['unit_price']

# Display required columns
print(df[['customer_name', 'product', 'quantity', 'unit_price', 'revenue']])

# Total revenue
print(f'\nTotal Revenue: {df["revenue"].sum()}')

       customer_name     product  quantity  unit_price  revenue
0       Ramesh Kumar      Laptop         2       45000    90000
1         Priya Nair         NaN         1       15000    15000
2         Amit Verma    Keyboard         3        1200     3600
3       Sunita Patel     Monitor         2       22000    44000
4       Ramesh Kumar      Laptop         2       45000    90000
6       Deepak Singh  Headphones         2        3500     7000
7   Unknown Customer      Webcam         1        2500     2500
8         Ananya Das      Laptop         1       45000    45000
9        Vikram Iyer    Keyboard         5        1200     6000
10       Pooja Gupta     Monitor         2       22000    44000
11        Suresh Rao     USB Hub         8         600     4800
12       Meera Joshi      Laptop         2       45000    90000
14       Tanvi Mehta      Webcam         2        2500     5000
15       Kiran Mehta       Mouse        10         800     8000
16      Deepak Singh    Keyboard        